In [1]:
pip install sacrebleu evaluate sacremoses

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.5.1
    Uninstalling fsspec-2025.5.1:
      Successfully uninstalled fsspec-2025.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Lin

In [8]:
import os
import torch
from datasets import load_dataset
from transformers import MarianMTModel, MarianTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
from datetime import datetime
import evaluate
import numpy as np

In [17]:
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_DATASETS_CACHE"] = "./hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "./transformers_cache"
os.environ["WANDB_DISABLED"] = "true"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU only")

raw_datasets = load_dataset("wmt14", "fr-en")

model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

def has_both_languages(example):
    t = example.get("translation")
    return t is not None and t.get("en") is not None and t.get("fr") is not None

clean_train_dataset = raw_datasets["train"].filter(has_both_languages).shuffle(seed=42)
small_train_dataset = clean_train_dataset.select(range(100000))
small_val_dataset = clean_train_dataset.select(range(100000, 125000))

def preprocess_function(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True, padding="max_length")
    return model_inputs

train_dataset = small_train_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])
val_dataset = small_val_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")

bleu = evaluate.load("sacrebleu")
meteor = evaluate.load("meteor")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    bleu_score = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    meteor_score = meteor.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": bleu_score["score"], "meteor": meteor_score["meteor"]}

CUDA available: True
Using GPU: Tesla P100-PCIE-16GB


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/30 [00:00<?, ?it/s]

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [18]:
training_args = Seq2SeqTrainingArguments(
    run_name=f"marianmt-wmt14-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}",
    output_dir="./outputs_marianmt_wmt14",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),
    num_train_epochs=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_steps=50,
    save_total_limit=1,
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipykernel_36/4257672725.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
print("Starting training...")
trainer.train()
trainer.save_model("./translation1")
tokenizer.save_pretrained("./translation1")
!zip -r translation1.zip ./translation1

Starting training...


Epoch,Training Loss,Validation Loss


In [4]:
import os
import torch
from datasets import load_dataset
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    PrinterCallback,
)
from datetime import datetime
import evaluate
import numpy as np
import time

# Silence warnings and set local cache paths
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_DATASETS_CACHE"] = "./hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "./transformers_cache"

# Show device info
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU only")

# Load dataset
print("Loading dataset...")
raw_datasets = load_dataset("wmt14", "fr-en")

# Filter out bad samples
def has_both_languages(example):
    t = example.get("translation")
    return t is not None and t.get("en") is not None and t.get("fr") is not None

clean_train_dataset = raw_datasets["train"].filter(has_both_languages).shuffle(seed=42)

# Use smaller subset
print("Selecting 10K samples for train, 2.5K for val")
small_train_dataset = clean_train_dataset.select(range(50000))
small_val_dataset = clean_train_dataset.select(range(50000, 60000))

# Load tokenizer and model
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Preprocessing
def preprocess_function(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    return tokenizer(inputs, text_target=targets, max_length=128, truncation=True, padding="max_length")

print("Tokenizing datasets...")
start = time.time()
train_dataset = small_train_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])
val_dataset = small_val_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])
print("Tokenization completed in", round(time.time() - start, 2), "seconds.")

# Set PyTorch format
train_dataset.set_format("torch")
val_dataset.set_format("torch")

# Load evaluation metrics
bleu = evaluate.load("sacrebleu")
meteor = evaluate.load("meteor")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    bleu_score = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    meteor_score = meteor.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": bleu_score["score"], "meteor": meteor_score["meteor"]}

# Training arguments
training_args = Seq2SeqTrainingArguments(
    run_name=f"marianmt-wmt14-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}",
    output_dir="./outputs_marianmt_wmt14",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),
    num_train_epochs=15,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    save_total_limit=1,
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    disable_tqdm=False,
)

# Trainer setup
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[PrinterCallback(), EarlyStoppingCallback(early_stopping_patience=1)],
)

# Train with error handling
try:
    print("Training started...")
    trainer.train()
    print("Training complete.")
except Exception as e:
    print("Training failed:", e)

# Save model
trainer.save_model("./translation1")
tokenizer.save_pretrained("./translation1")

# Zip the folder for download
!zip -r translation1.zip ./translation1


2025-07-14 16:02:19.068927: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752508939.253824      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752508939.307030      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


CUDA available: True
Using GPU: Tesla P100-PCIE-16GB
Loading dataset...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

train-00000-of-00030.parquet:   0%|          | 0.00/252M [00:00<?, ?B/s]

...

{'loss': 4.4347, 'grad_norm': 51.28126907348633, 'learning_rate': 1.99957346982299e-05, 'epoch': 0.0064}
{'loss': 0.5808, 'grad_norm': 2.44006085395813, 'learning_rate': 1.99872040946897e-05, 'epoch': 0.0128}
{'loss': 0.4758, 'grad_norm': 1.3953168392181396, 'learning_rate': 1.99786734911495e-05, 'epoch': 0.0192}
{'loss': 0.4566, 'grad_norm': 0.718425452709198, 'learning_rate': 1.99701428876093e-05, 'epoch': 0.0256}
{'loss': 0.4544, 'grad_norm': 0.8789302706718445, 'learning_rate': 1.99616122840691e-05, 'epoch': 0.032}
{'loss': 0.4309, 'grad_norm': 1.1187156438827515, 'learning_rate': 1.9953081680528898e-05, 'epoch': 0.0384}
{'loss': 0.4797, 'grad_norm': 0.7636718153953552, 'learning_rate': 1.99445510769887e-05, 'epoch': 0.0448}
{'loss': 0.3897, 'grad_norm': 0.6730805039405823, 'learning_rate': 1.9936020473448498e-05, 'epoch': 0.0512}
{'loss': 0.4813, 'grad_norm': 0.7578197121620178, 'learning_rate': 1.9918959266368096e-05, 'epoch': 0.064}
{'loss': 0.4335, 'grad_norm': 0.86715716123580

/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[59513]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


{'loss': 0.3024, 'grad_norm': 0.5835391879081726, 'learning_rate': 1.866496054595863e-05, 'epoch': 1.00448}
{'loss': 0.3652, 'grad_norm': 0.5815303325653076, 'learning_rate': 1.8656429942418428e-05, 'epoch': 1.01088}
{'loss': 0.3753, 'grad_norm': 0.7255005240440369, 'learning_rate': 1.864789933887823e-05, 'epoch': 1.01728}
{'loss': 0.4183, 'grad_norm': 0.7784653306007385, 'learning_rate': 1.8639368735338025e-05, 'epoch': 1.02368}
{'loss': 0.3164, 'grad_norm': 0.6801709532737732, 'learning_rate': 1.8630838131797827e-05, 'epoch': 1.03008}
{'loss': 0.3972, 'grad_norm': 0.7582751512527466, 'learning_rate': 1.8622307528257626e-05, 'epoch': 1.03648}
{'loss': 0.3638, 'grad_norm': 0.6839199066162109, 'learning_rate': 1.8613776924717428e-05, 'epoch': 1.04288}
{'loss': 0.3518, 'grad_norm': 0.7743842601776123, 'learning_rate': 1.8605246321177223e-05, 'epoch': 1.04928}
{'loss': 0.3488, 'grad_norm': 0.5983806252479553, 'learning_rate': 1.8596715717637025e-05, 'epoch': 1.05568}
{'loss': 0.3859, 'gra

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


{'train_runtime': 5464.6528, 'train_samples_per_second': 137.246, 'train_steps_per_second': 4.29, 'train_loss': 0.39694148752068525, 'epoch': 2.0}
Training complete.
  adding: translation1/ (stored 0%)
  adding: translation1/model.safetensors (deflated 7%)
  adding: translation1/target.spm (deflated 50%)
  adding: translation1/vocab.json (deflated 70%)
  adding: translation1/source.spm (deflated 49%)
  adding: translation1/special_tokens_map.json (deflated 35%)
  adding: translation1/generation_config.json (deflated 43%)
  adding: translation1/tokenizer_config.json (deflated 68%)
  adding: translation1/config.json (deflated 62%)
  adding: translation1/training_args.bin (deflated 51%)


In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
def preprocess_function(examples):
    inputs = ["translate English to French: " + ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    return tokenizer(inputs, text_target=targets, max_length=128, truncation=True, padding="max_length")

model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

print("Tokenizing datasets...")
start = time.time()
train_dataset = small_train_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])
val_dataset = small_val_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])
print("Tokenization completed in", round(time.time() - start, 2), "seconds.")

# Set PyTorch format
train_dataset.set_format("torch")
val_dataset.set_format("torch")

# Load evaluation metrics
bleu = evaluate.load("sacrebleu")
meteor = evaluate.load("meteor")

# Training arguments
training_args = Seq2SeqTrainingArguments(
    run_name=f"marianmt-wmt14-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}",
    output_dir="./outputs_marianmt_wmt14",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),
    num_train_epochs=15,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    save_total_limit=1,
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    disable_tqdm=False,
)

# Trainer setup
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[PrinterCallback(), EarlyStoppingCallback(early_stopping_patience=1)],
)

# Train with error handling
try:
    print("Training started...")
    trainer.train()
    print("Training complete.")
except Exception as e:
    print("Training failed:", e)

# Save model
trainer.save_model("./translation1")
tokenizer.save_pretrained("./translation1")

# Zip the folder for download
!zip -r translation1.zip ./translation1


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
/tmp/ipykernel_36/29487619.py:49: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Training started...


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Bleu,Meteor
1,0.454700,0.391671,7.343758,0.302399
2,0.453700,0.389444,7.411311,0.303635
3,0.431400,0.388693,7.405455,0.303532
4,0.470000,0.387745,7.426476,0.303941


{'loss': 10.6487, 'grad_norm': 32.327476501464844, 'learning_rate': 1.9992322456813822e-05, 'epoch': 0.0064}
{'loss': 9.5422, 'grad_norm': 30.58890151977539, 'learning_rate': 1.998379185327362e-05, 'epoch': 0.0128}
{'loss': 8.2756, 'grad_norm': 37.54740524291992, 'learning_rate': 1.997526124973342e-05, 'epoch': 0.0192}
{'loss': 7.6107, 'grad_norm': 41.42388916015625, 'learning_rate': 1.996673064619322e-05, 'epoch': 0.0256}
{'loss': 6.3249, 'grad_norm': 43.31626892089844, 'learning_rate': 1.995820004265302e-05, 'epoch': 0.032}
{'loss': 5.3355, 'grad_norm': 40.30341339111328, 'learning_rate': 1.994966943911282e-05, 'epoch': 0.0384}
{'loss': 4.1105, 'grad_norm': 40.976444244384766, 'learning_rate': 1.9941138835572617e-05, 'epoch': 0.0448}
{'loss': 3.4497, 'grad_norm': 35.919586181640625, 'learning_rate': 1.993260823203242e-05, 'epoch': 0.0512}
{'loss': 2.622, 'grad_norm': 29.53961944580078, 'learning_rate': 1.9924077628492218e-05, 'epoch': 0.0576}
{'loss': 1.9615, 'grad_norm': 13.74830436

In [2]:
import os
import torch
from datasets import load_dataset
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    PrinterCallback,
)
from datetime import datetime
import evaluate
import numpy as np
import time

# Silence warnings and set local cache paths
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_DATASETS_CACHE"] = "./hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "./transformers_cache"

# Show device info
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU only")

# Load dataset
print("Loading dataset...")
raw_datasets = load_dataset("wmt14", "fr-en")

# Filter out bad samples
def has_both_languages(example):
    t = example.get("translation")
    return t is not None and t.get("en") is not None and t.get("fr") is not None

clean_train_dataset = raw_datasets["train"].filter(has_both_languages).shuffle(seed=42)

# Use smaller subset
print("Selecting 10K samples for train, 2.5K for val")
small_train_dataset = clean_train_dataset.select(range(50000))
small_val_dataset = clean_train_dataset.select(range(50000, 60000))

# Load tokenizer and model
model_name = "Helsinki-NLP/opus-mt-en-ROMANCE"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Preprocessing
def preprocess_function(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    return tokenizer(inputs, text_target=targets, max_length=128, truncation=True, padding="max_length")

print("Tokenizing datasets...")
start = time.time()
train_dataset = small_train_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])
val_dataset = small_val_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])
print("Tokenization completed in", round(time.time() - start, 2), "seconds.")

# Set PyTorch format
train_dataset.set_format("torch")
val_dataset.set_format("torch")

# Load evaluation metrics
bleu = evaluate.load("sacrebleu")
meteor = evaluate.load("meteor")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    bleu_score = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    meteor_score = meteor.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": bleu_score["score"], "meteor": meteor_score["meteor"]}

# Training arguments
training_args = Seq2SeqTrainingArguments(
    run_name=f"marianmt-wmt14-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}",
    output_dir="./outputs_marianmt_wmt14",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),
    num_train_epochs=15,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    save_total_limit=1,
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    disable_tqdm=False,
)

# Trainer setup
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[PrinterCallback(), EarlyStoppingCallback(early_stopping_patience=1)],
)

# Train with error handling
try:
    print("Training started...")
    trainer.train()
    print("Training complete.")
except Exception as e:
    print("Training failed:", e)

# Save model
trainer.save_model("./translation2")
tokenizer.save_pretrained("./translation2")

# Zip the folder for download
!zip -r translation2.zip ./translation2


2025-07-16 11:33:33.179751: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752665613.378290      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752665613.434185      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


CUDA available: True
Using GPU: Tesla P100-PCIE-16GB
Loading dataset...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

train-00000-of-00030.parquet:   0%|          | 0.00/252M [00:00<?, ?B/s]

...

{'loss': 3.9979, 'grad_norm': 5.949398994445801, 'learning_rate': 1.999488163787588e-05, 'epoch': 0.0064}
{'loss': 0.5424, 'grad_norm': 2.6062169075012207, 'learning_rate': 1.998635103433568e-05, 'epoch': 0.0128}
{'loss': 0.4858, 'grad_norm': 3.4362759590148926, 'learning_rate': 1.997782043079548e-05, 'epoch': 0.0192}
{'loss': 0.4875, 'grad_norm': 1.052423357963562, 'learning_rate': 1.996928982725528e-05, 'epoch': 0.0256}
{'loss': 0.4867, 'grad_norm': 1.1950887441635132, 'learning_rate': 1.996075922371508e-05, 'epoch': 0.032}
{'loss': 0.4645, 'grad_norm': 2.7965645790100098, 'learning_rate': 1.995222862017488e-05, 'epoch': 0.0384}
{'loss': 0.531, 'grad_norm': 11.769576072692871, 'learning_rate': 1.9943698016634678e-05, 'epoch': 0.0448}
{'loss': 0.4188, 'grad_norm': 0.952887773513794, 'learning_rate': 1.9935167413094477e-05, 'epoch': 0.0512}
{'loss': 0.4498, 'grad_norm': 1.0345996618270874, 'learning_rate': 1.992663680955428e-05, 'epoch': 0.0576}
{'loss': 0.5054, 'grad_norm': 1.11867761

/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[65000]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


{'loss': 0.3249, 'grad_norm': 0.8182711601257324, 'learning_rate': 1.8664107485604608e-05, 'epoch': 1.00448}
{'loss': 0.3977, 'grad_norm': 0.862136960029602, 'learning_rate': 1.8655576882064406e-05, 'epoch': 1.01088}
{'loss': 0.4025, 'grad_norm': 1.0217255353927612, 'learning_rate': 1.8647046278524208e-05, 'epoch': 1.01728}
{'loss': 0.4474, 'grad_norm': 1.0677366256713867, 'learning_rate': 1.8638515674984007e-05, 'epoch': 1.02368}
{'loss': 0.3418, 'grad_norm': 1.033261775970459, 'learning_rate': 1.8629985071443806e-05, 'epoch': 1.03008}
{'loss': 0.4339, 'grad_norm': 1.040164589881897, 'learning_rate': 1.8621454467903604e-05, 'epoch': 1.03648}
{'loss': 0.3967, 'grad_norm': 0.94209223985672, 'learning_rate': 1.8612923864363406e-05, 'epoch': 1.04288}
{'loss': 0.3818, 'grad_norm': 0.9306655526161194, 'learning_rate': 1.8604393260823205e-05, 'epoch': 1.04928}
{'loss': 0.3729, 'grad_norm': 0.8551579713821411, 'learning_rate': 1.8595862657283004e-05, 'epoch': 1.05568}
{'loss': 0.4179, 'grad_n

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


{'train_runtime': 7267.0153, 'train_samples_per_second': 103.206, 'train_steps_per_second': 3.226, 'train_loss': 0.4243896421681439, 'epoch': 2.0}
Training complete.
  adding: translation2/ (stored 0%)
  adding: translation2/generation_config.json (deflated 43%)
  adding: translation2/config.json (deflated 63%)
  adding: translation2/special_tokens_map.json (deflated 35%)
  adding: translation2/tokenizer_config.json (deflated 62%)
  adding: translation2/training_args.bin (deflated 51%)
  adding: translation2/model.safetensors (deflated 8%)
  adding: translation2/source.spm (deflated 50%)
  adding: translation2/vocab.json (deflated 70%)
  adding: translation2/target.spm (deflated 49%)
